<a href="https://colab.research.google.com/github/hetaf234/GP_Layers/blob/main/image_recognition_model_using_food101_(MobileNetV3).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Connecting to google drive

In [11]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Verifying the paths & checking partitions

In [12]:
import os

# Adjust this path to match exactly where you uploaded the Food-101 folder
FOOD101_PATH = '/content/drive/MyDrive/Graduation_Project/food-101/food-101/images'

print("Does the path exist?", os.path.exists(FOOD101_PATH))

Does the path exist? True


In [13]:
print(os.listdir(FOOD101_PATH))

['.DS_Store', 'waffles', 'tuna_tartare', 'tacos', 'takoyaki', 'strawberry_shortcake', 'tiramisu', 'steak', 'seaweed_salad', 'spring_rolls', 'shrimp_and_grits', 'spaghetti_carbonara', 'sushi', 'spaghetti_bolognese', 'sashimi', 'scallops', 'samosa', 'ramen', 'pork_chop', 'peking_duck', 'red_velvet_cake', 'pho', 'pancakes', 'prime_rib', 'ravioli', 'poutine', 'pulled_pork_sandwich', 'panna_cotta', 'pizza', 'risotto', 'paella', 'oysters', 'pad_thai', 'omelette', 'lobster_bisque', 'miso_soup', 'macaroni_and_cheese', 'macarons', 'onion_rings', 'mussels', 'hummus', 'lobster_roll_sandwich', 'ice_cream', 'nachos', 'lasagna', 'huevos_rancheros', 'hamburger', 'hot_dog', 'hot_and_sour_soup', 'grilled_salmon', 'gnocchi', 'french_onion_soup', 'french_fries', 'greek_salad', 'grilled_cheese_sandwich', 'guacamole', 'garlic_bread', 'gyoza', 'fried_rice', 'frozen_yogurt', 'fried_calamari', 'french_toast', 'fish_and_chips', 'filet_mignon', 'foie_gras', 'donuts', 'falafel', 'dumplings', 'deviled_eggs', 'cre

In [14]:
# Count how many class folders exist (excluding hidden system files like .DS_Store)
class_folders = [f for f in os.listdir(FOOD101_PATH) if not f.startswith('.')]

print(f"Number of class folders: {len(class_folders)}")
print(f"First 5 classes: {class_folders[:5]}")

Number of class folders: 101
First 5 classes: ['waffles', 'tuna_tartare', 'tacos', 'takoyaki', 'strawberry_shortcake']


In [6]:
total_images = 0
for class_name in class_folders:
    class_path = os.path.join(FOOD101_PATH, class_name)
    num_images = len([f for f in os.listdir(class_path) if f.endswith('.jpg')])
    total_images += num_images

print(f"Total images across all classes: {total_images}")
print(f"Average images per class: {total_images / len(class_folders):.1f}")

Total images across all classes: 101118
Average images per class: 1001.2


In [15]:
parent_path = '/content/drive/MyDrive/Graduation_Project/food-101/food-101'
print(os.listdir(parent_path))

['README.txt', 'license_agreement.txt', '.DS_Store', 'meta', 'images']


In [16]:
meta_path = '/content/drive/MyDrive/Graduation_Project/food-101/food-101/meta'
print(os.listdir(meta_path))

['test.json', 'labels.txt', 'test.txt', 'train.json', 'classes.txt', 'train.txt']


In [17]:
LOCAL_FOOD101_PATH = '/content/food-101-local'

In [18]:
import shutil, time
import os

LOCAL_FOOD101_PATH = '/content/food-101-local'

# Remove the directory if it already exists to prevent FileExistsError
if os.path.exists(LOCAL_FOOD101_PATH):
    print(f"Removing existing directory: {LOCAL_FOOD101_PATH}")
    shutil.rmtree(LOCAL_FOOD101_PATH)

print("Copying images to local disk...")
start = time.time()
shutil.copytree(FOOD101_PATH, LOCAL_FOOD101_PATH)
print(f"Copy completed in {(time.time()-start)/60:.1f} minutes")

Removing existing directory: /content/food-101-local
Copying images to local disk...
Copy completed in 10.2 minutes


In [19]:
# Read the split file (train.txt or test.txt), where each line looks like "class_name/image_id"
# Split each line into class name and image ID, then build the full file path for that image
def load_split(split_file, images_root):
    with open(split_file) as f:
        entries = f.read().splitlines()

    image_paths = []
    labels = []

    for entry in entries:
        class_name, image_id = entry.split('/')
        full_path = os.path.join(images_root, class_name, image_id + '.jpg')
        image_paths.append(full_path)
        labels.append(class_name)

    return image_paths, labels

# Apply this function once for the train split, and once for the test split
# NOTE: using LOCAL_FOOD101_PATH instead of FOOD101_PATH for faster access
train_paths, train_labels = load_split(os.path.join(meta_path, 'train.txt'), LOCAL_FOOD101_PATH)
test_paths, test_labels = load_split(os.path.join(meta_path, 'test.txt'), LOCAL_FOOD101_PATH)

print("Done loading paths and labels")
print(f"Train samples: {len(train_paths)}")
print(f"Test samples: {len(test_paths)}")

Done loading paths and labels
Train samples: 75750
Test samples: 25250


In [20]:
from collections import Counter

train_counts = Counter(train_labels)
test_counts = Counter(test_labels)

print("Train - min per class:", min(train_counts.values()))
print("Train - max per class:", max(train_counts.values()))
print("Test - min per class:", min(test_counts.values()))
print("Test - max per class:", max(test_counts.values()))

Train - min per class: 750
Train - max per class: 750
Test - min per class: 250
Test - max per class: 250


# Train/ Val split

In [21]:
# Food-101 only provides train/test officially - we need to carve out a validation set from train
from sklearn.model_selection import train_test_split

# Split 85% for actual training, 15% for validation, keeping class balance (stratify)
train_paths_final, val_paths, train_labels_final, val_labels = train_test_split(
    train_paths, train_labels,
    test_size=0.15,
    stratify=train_labels,  # ensures each class is proportionally represented in both splits
    random_state=42
)

print(f"Final train samples: {len(train_paths_final)}")
print(f"Validation samples: {len(val_paths)}")

Final train samples: 64387
Validation samples: 11363


# Custom Dataset Class

In [22]:
# Read the list of 101 class names from classes.txt
with open(os.path.join(meta_path, 'classes.txt')) as f:
    classes_list = f.read().splitlines()

print(f"Number of classes: {len(classes_list)}")
print(f"First 5: {classes_list[:5]}")

Number of classes: 101
First 5: ['apple_pie', 'baby_back_ribs', 'baklava', 'beef_carpaccio', 'beef_tartare']


In [23]:
# Build a mapping from class name to a numeric index (needed by the model)
class_to_idx = {class_name: idx for idx, class_name in enumerate(sorted(classes_list))}

print(f"Number of classes: {len(class_to_idx)}")
print(f"Example mapping: {list(class_to_idx.items())[:3]}")

Number of classes: 101
Example mapping: [('apple_pie', 0), ('baby_back_ribs', 1), ('baklava', 2)]


In [24]:
# Custom PyTorch Dataset class that reads images from their file paths
# and returns them along with their numeric class label
from torch.utils.data import Dataset
from PIL import Image

class Food101Dataset(Dataset):
    def __init__(self, image_paths, labels, class_to_idx, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.class_to_idx = class_to_idx  # maps class name (string) to a number
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, index):
        image_path = self.image_paths[index]
        label_name = self.labels[index]

        image = Image.open(image_path).convert('RGB')
        if self.transform:
            image = self.transform(image)

        label_idx = self.class_to_idx[label_name]
        return image, label_idx

print("Food101Dataset class defined successfully")

Food101Dataset class defined successfully


# Transforms

In [25]:
# Transformations applied to training images: resize, augment, normalize
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),  # randomly crop a region (70%-100% of image) and resize to 224x224
    transforms.RandomHorizontalFlip(p=0.5),                 # randomly flip image horizontally
    transforms.RandomRotation(15),                          # randomly rotate up to 15 degrees
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),  # vary lighting/color
    transforms.ToTensor(),                                  # convert image to a PyTorch tensor
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),  # match ImageNet stats
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.15))     # randomly erase a small patch (helps model not rely on one region)
])

# Transformations for validation/test images: NO augmentation, just resize + normalize
test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("Transforms defined successfully")

Transforms defined successfully


# Data Loaders

In [26]:
# Build the actual Dataset objects using our custom class + transforms
train_dataset = Food101Dataset(train_paths_final, train_labels_final, class_to_idx, transform=train_transform)
val_dataset = Food101Dataset(val_paths, val_labels, class_to_idx, transform=test_transform)
test_dataset = Food101Dataset(test_paths, test_labels, class_to_idx, transform=test_transform)

print(f"Train dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")

Train dataset size: 64387
Validation dataset size: 11363
Test dataset size: 25250


In [27]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)



print("DataLoaders created successfully")
print(f"Number of training batches: {len(train_loader)}")
print(f"Number of validation batches: {len(val_loader)}")
print(f"Number of test batches: {len(test_loader)}")

DataLoaders created successfully
Number of training batches: 2013
Number of validation batches: 356
Number of test batches: 790


# Starting MobileNetV3 Model

In [28]:
# Load a pretrained MobileNetV3-Large model and adapt it for our 101 classes
from torchvision.models import mobilenet_v3_large, MobileNet_V3_Large_Weights
import torch
import torch.nn as nn

# Load the model with weights pretrained on ImageNet
model = mobilenet_v3_large(weights=MobileNet_V3_Large_Weights.DEFAULT)

# Freeze all layers - we don't want to retrain the general visual knowledge
for param in model.parameters():
    param.requires_grad = False

# Function to fully freeze BatchNorm layers (weights, bias, and running stats)
def freeze_bn(module):
    for m in module.modules():
        if isinstance(m, nn.BatchNorm2d):
            m.eval()                        # stop updating running_mean/running_var
            m.weight.requires_grad = False
            m.bias.requires_grad = False

# Apply BatchNorm freezing to the whole feature extractor
freeze_bn(model.features)

# Replace the final classification layer to output 101 classes instead of 1000
model.classifier[3] = nn.Linear(model.classifier[3].in_features, 101)

# Move the model to GPU if available, otherwise use CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

print(f"Using device: {device}")
print("Model ready with 101 output classes")

Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-5c1a4163.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_large-5c1a4163.pth


100%|██████████| 21.1M/21.1M [00:00<00:00, 83.1MB/s]


Using device: cuda
Model ready with 101 output classes


In [29]:
import torch.optim as optim

# Label smoothing helps the model not be overconfident, especially useful since Food-101 has some noisy labels
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = optim.Adam(model.classifier[3].parameters(), lr=0.001, weight_decay=1e-4)

scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

print("Training tools ready: criterion, optimizer, scheduler")

Training tools ready: criterion, optimizer, scheduler


In [30]:
# Function that runs one full training epoch: shows the model all training images once
from tqdm import tqdm

# Re-freezes BatchNorm layers after model.train() (since model.train() re-enables them by default)
def set_train_mode(model):
    model.train()
    for m in model.modules():
        if isinstance(m, nn.BatchNorm2d) and not m.weight.requires_grad:
            m.eval()  # keep frozen BN layers in eval mode

def train_one_epoch(model, loader, criterion, optimizer, device):
    set_train_mode(model)  # switch to training mode, but keep frozen BN layers frozen
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in tqdm(loader, desc="Training"):
        images, labels = images.to(device), labels.to(device)  # move batch to GPU

        optimizer.zero_grad()               # clear old gradients
        outputs = model(images)              # forward pass: model makes predictions
        loss = criterion(outputs, labels)    # calculate how wrong the predictions are
        loss.backward()                      # backward pass: calculate how to adjust weights
        optimizer.step()                     # apply the weight update

        running_loss += loss.item()
        _, predicted = outputs.max(1)        # get the predicted class for each image
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    epoch_loss = running_loss / len(loader)
    epoch_acc = 100. * correct / total
    return epoch_loss, epoch_acc


# Function that checks model performance on validation data (no learning happens here)
def validate(model, loader, criterion, device):
    model.eval()  # switch model to evaluation mode (disables dropout, etc.)
    running_loss, correct, total = 0.0, 0, 0

    with torch.no_grad():  # no need to track gradients during validation
        for images, labels in tqdm(loader, desc="Validating"):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    epoch_loss = running_loss / len(loader)
    epoch_acc = 100. * correct / total
    return epoch_loss, epoch_acc

print("Training and validation functions defined")

Training and validation functions defined


In [31]:
# Set up the folder where we'll save model checkpoints and results
PROJECT_PATH = '/content/drive/MyDrive/Graduation_Project/food101_results'
os.makedirs(PROJECT_PATH, exist_ok=True)

NUM_EPOCHS = 15
best_acc = 0.0

# Early stopping settings: stop if val accuracy doesn't improve for this many epochs
PATIENCE = 4
epochs_no_improve = 0

# Lists to store metrics from each epoch, used later for plotting
train_losses, val_losses = [], []
train_accs, val_accs = [], []

for epoch in range(NUM_EPOCHS):
    print(f"\n--- Epoch {epoch+1}/{NUM_EPOCHS} ---")

    # Train for one full epoch
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)

    # Check performance on validation set (model doesn't learn from this)
    val_loss, val_acc = validate(model, val_loader, criterion, device)

    # Reduce learning rate according to schedule
    scheduler.step()

    # Store results for plotting later
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")

    # Always save the latest checkpoint (protects against session interruption)
    torch.save(model.state_dict(), os.path.join(PROJECT_PATH, 'last_checkpoint.pth'))

    # Save the best model separately, only when validation accuracy improves
    if val_acc > best_acc:
        best_acc = val_acc
        epochs_no_improve = 0
        torch.save(model.state_dict(), os.path.join(PROJECT_PATH, 'best_model.pth'))
        print("✅ Best model saved")
    else:
        epochs_no_improve += 1
        print(f"No improvement for {epochs_no_improve} epoch(s)")

    # Stop early if no improvement for PATIENCE consecutive epochs
    if epochs_no_improve >= PATIENCE:
        print(f"\n⏹️ Early stopping triggered after {epoch+1} epochs")
        break

print("\nTraining complete!")
print(f"Best validation accuracy achieved: {best_acc:.2f}%")


--- Epoch 1/15 ---


Validating: 100%|██████████| 356/356 [00:29<00:00, 12.21it/s]


Train Loss: 2.5340 | Train Acc: 51.03%
Val Loss: 2.1571 | Val Acc: 61.03%
✅ Best model saved

--- Epoch 2/15 ---


Validating: 100%|██████████| 356/356 [00:29<00:00, 12.22it/s]


Train Loss: 2.2239 | Train Acc: 58.61%
Val Loss: 2.1088 | Val Acc: 61.97%
✅ Best model saved

--- Epoch 3/15 ---


...

Train Loss: 1.9997 | Train Acc: 65.76%
Val Loss: 2.0162 | Val Acc: 65.13%
No improvement for 2 epoch(s)

--- Epoch 13/15 ---


Validating: 100%|██████████| 356/356 [00:28<00:00, 12.28it/s]


Train Loss: 2.0004 | Train Acc: 65.53%
Val Loss: 2.0158 | Val Acc: 65.10%
No improvement for 3 epoch(s)

--- Epoch 14/15 ---


Validating: 100%|██████████| 356/356 [00:28<00:00, 12.30it/s]

Train Loss: 2.0001 | Train Acc: 65.58%
Val Loss: 2.0156 | Val Acc: 65.13%
No improvement for 4 epoch(s)

⏹️ Early stopping triggered after 14 epochs

Training complete!
Best validation accuracy achieved: 65.25%


In [32]:
# Load the best model from the previous training round before fine-tuning
model.load_state_dict(torch.load(os.path.join(PROJECT_PATH, 'best_model.pth')))
print("Loaded best model from previous training (65.25% val accuracy)")

Loaded best model from previous training (65.25% val accuracy)


In [33]:
# Unfreeze a larger portion of the feature extractor (last 6 blocks instead of 3)
# so the model can learn deeper food-specific features, not just the very last layer
for param in model.features[-6:].parameters():
    param.requires_grad = True

# Also unfreeze the full classifier (not just the last layer) so it can adapt further
for param in model.classifier.parameters():
    param.requires_grad = True

# Re-apply BN freezing logic: only let BatchNorm layers that belong to the
# newly unfrozen blocks actually train; keep the rest fully frozen
def unfreeze_bn(module):
    for m in module.modules():
        if isinstance(m, nn.BatchNorm2d):
            m.train()
            m.weight.requires_grad = True
            m.bias.requires_grad = True

unfreeze_bn(model.features[-6:])

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable_params:,} / {total_params:,}")

Trainable parameters: 4,139,469 / 4,331,413


In [34]:
# Use different learning rates for classifier vs feature layers
# Classifier gets a higher LR since it's learning food-specific decisions
# Feature layers get a much smaller LR to avoid destroying pretrained knowledge
optimizer_finetune = optim.Adam([
    {'params': model.classifier.parameters(), 'lr': 1e-4},
    {'params': model.features[-6:].parameters(), 'lr': 1e-5}
], weight_decay=1e-4)

# Cosine annealing gives a smoother, more gradual LR decay than step-wise drops
scheduler_finetune = optim.lr_scheduler.CosineAnnealingLR(optimizer_finetune, T_max=20)

print("Fine-tuning optimizer ready")

Fine-tuning optimizer ready


In [35]:
NUM_EPOCHS_FINETUNE = 20  # increased since we're training more layers now, they need more time to adapt
best_acc_finetune = best_acc

# Early stopping settings for fine-tuning stage
PATIENCE_FT = 5
epochs_no_improve_ft = 0

train_losses_ft, val_losses_ft = [], []
train_accs_ft, val_accs_ft = [], []

for epoch in range(NUM_EPOCHS_FINETUNE):
    print(f"\n--- Fine-tune Epoch {epoch+1}/{NUM_EPOCHS_FINETUNE} ---")

    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer_finetune, device)
    val_loss, val_acc = validate(model, val_loader, criterion, device)
    scheduler_finetune.step()

    train_losses_ft.append(train_loss)
    val_losses_ft.append(val_loss)
    train_accs_ft.append(train_acc)
    val_accs_ft.append(val_acc)

    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")

    torch.save(model.state_dict(), os.path.join(PROJECT_PATH, 'last_checkpoint_finetuned.pth'))

    if val_acc > best_acc_finetune:
        best_acc_finetune = val_acc
        epochs_no_improve_ft = 0
        torch.save(model.state_dict(), os.path.join(PROJECT_PATH, 'best_model_finetuned.pth'))
        print("✅ Best fine-tuned model saved")
    else:
        epochs_no_improve_ft += 1
        print(f"No improvement for {epochs_no_improve_ft} epoch(s)")

    if epochs_no_improve_ft >= PATIENCE_FT:
        print(f"\n⏹️ Early stopping triggered after {epoch+1} epochs")
        break

print("\nFine-tuning complete!")
print(f"Best validation accuracy after fine-tuning: {best_acc_finetune:.2f}%")
print(f"Improvement over feature-extraction: {best_acc_finetune - 65.25:.2f}%")


--- Fine-tune Epoch 1/20 ---


Validating: 100%|██████████| 356/356 [00:29<00:00, 12.13it/s]


Train Loss: 2.3350 | Train Acc: 55.77%
Val Loss: 2.0279 | Val Acc: 65.11%
No improvement for 1 epoch(s)

--- Fine-tune Epoch 2/20 ---


Validating: 100%|██████████| 356/356 [00:29<00:00, 12.27it/s]


Train Loss: 2.0084 | Train Acc: 65.15%
Val Loss: 1.9028 | Val Acc: 68.93%
✅ Best fine-tuned model saved

--- Fine-tune Epoch 3/20 ---


...

Train Loss: 1.4714 | Train Acc: 82.39%
Val Loss: 1.6729 | Val Acc: 75.39%
✅ Best fine-tuned model saved

--- Fine-tune Epoch 19/20 ---


Validating: 100%|██████████| 356/356 [00:29<00:00, 11.88it/s]


Train Loss: 1.4679 | Train Acc: 82.49%
Val Loss: 1.6719 | Val Acc: 75.46%
✅ Best fine-tuned model saved

--- Fine-tune Epoch 20/20 ---


Validating: 100%|██████████| 356/356 [00:29<00:00, 12.02it/s]

Train Loss: 1.4674 | Train Acc: 82.53%
Val Loss: 1.6714 | Val Acc: 75.45%
No improvement for 1 epoch(s)

Fine-tuning complete!
Best validation accuracy after fine-tuning: 75.46%
Improvement over feature-extraction: 10.21%


In [36]:
import os

# Save the current model with a descriptive filename (model name + accuracy)
descriptive_name = "MobileNetV3Large_finetuned_acc75.46.pth"
descriptive_path = os.path.join(PROJECT_PATH, descriptive_name)

torch.save(model.state_dict(), descriptive_path)

file_size_mb = os.path.getsize(descriptive_path) / (1024 * 1024)
print(f"✅ Model saved as: {descriptive_name}")
print(f"📁 Full path: {descriptive_path}")
print(f"📦 File size: {file_size_mb:.1f} MB")

✅ Model saved as: MobileNetV3Large_finetuned_acc75.46.pth
📁 Full path: /content/drive/MyDrive/Graduation_Project/food101_results/MobileNetV3Large_finetuned_acc75.46.pth
📦 File size: 16.7 MB
